# 04. CIES Experiment: Credibility Index via Explanation Stability

## Tổng quan thí nghiệm CIES
Thí nghiệm đo lường độ ổn định của giải thích SHAP qua các lần bootstrap resample tập huấn luyện.

### Thuật toán bắt buộc (spec mục 7.1):
Với mỗi tổ hợp (Model × Imbalance Technique):
```
Cố định tập df_test_fixed_eval (đánh giá trên cùng mẫu eval)
Lặp N_RUNS lần (seed = 0 ... N_RUNS - 1):
  1. Bootstrap resample tập train
  2. Fit lại encoding từ đầu trên resample (KHÔNG dùng encoding cũ)
  3. Áp dụng kỹ thuật imbalance
  4. Huấn luyện mô hình
  5. Tính SHAP values trên df_test_fixed_eval
Tính CIES Score = 1 - Mean(Rank-Weighted Distance)
```

### Rank-Weighted Distance:
Trọng số $w_r = 1/r$ phạt nặng hơn khi các top features quan trọng bị hoán đổi thứ hạng.

In [ ]:
# 1. Setup & Imports
import sys
from pathlib import Path
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_DIR, RESULTS_DIR, SEED, N_RUNS, TARGET_COL,
    MODEL_NAMES, IMBALANCE_TECHNIQUES, ONEHOT_COLS, TARGET_ENCODE_COLS
)
from src.explainability.cies import run_cies_experiment_isolated, merge_cies_result

print(f'SEED: {SEED}, N_RUNS config: {N_RUNS}')
print('Models:', MODEL_NAMES)
print('Imbalance techniques:', IMBALANCE_TECHNIQUES)

In [ ]:
# 2. Chuẩn bị tập Train & Fixed Eval Set (RAW — chưa encode)
# QUAN TRỌNG: phải đọc bản RAW (train_raw.parquet / test_raw.parquet),
# KHÔNG phải *_encoded.parquet — vì run_cies_experiment() cần tự fit lại
# encoding từ đầu trên mỗi bootstrap resample ở mỗi run (spec mục 7.1).
# Nếu đọc *_encoded.parquet, các cột categorical gốc đã bị drop và bước
# re-encode bên trong sẽ thành no-op câm (dùng lại encoding cũ).
train_raw = pd.read_parquet(Path('../data/processed/train_raw.parquet'))
test_raw = pd.read_parquet(Path('../data/processed/test_raw.parquet'))

# Để đảm bảo tốc độ tính SHAP lặp lại N lần, lấy mẫu cố định eval set đại diện
# Bao gồm cả fraud và non-fraud
eval_fraud = test_raw[test_raw[TARGET_COL] == 1].sample(n=min(50, test_raw[TARGET_COL].sum()), random_state=SEED)
eval_legit = test_raw[test_raw[TARGET_COL] == 0].sample(n=min(200, (test_raw[TARGET_COL] == 0).sum()), random_state=SEED)
df_test_fixed_eval = pd.concat([eval_fraud, eval_legit]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f'Fixed Eval Set: {len(df_test_fixed_eval)} mẫu (Fraud: {df_test_fixed_eval[TARGET_COL].sum()})')
print('Eval set này sẽ được GIỮ NGUYÊN qua toàn bộ N_RUNS.')
# Cỡ mẫu train cho bootstrap. Train Sparkov đầy đủ (1.48M dòng) quá nặng cho 20 run/tổ hợp
# (SMOTE-ENN riêng phần resample ~1.2 giờ/run). 100.000 dòng ≈ 500 fraud: đủ để SMOTE và
# mô hình học từ vài trăm ví dụ fraud, thay vì ~60 ở 10.000 dòng (mẫu nhỏ làm CIES đo nhiễu
# chứ không đo ảnh hưởng của kỹ thuật imbalance). Lấy mẫu PHÂN TẦNG để giữ nguyên tỷ lệ fraud.
# Chạy mục 8 (kiểm tra độ nhạy) để có bằng chứng thực nghiệm cho con số này.
from sklearn.model_selection import train_test_split
SUBSAMPLE_N = 100_000

def stratified_subsample(df, n):
    if n >= len(df):
        return df
    sub, _ = train_test_split(df, train_size=n, stratify=df[TARGET_COL], random_state=SEED)
    return sub.reset_index(drop=True)

train_sub = stratified_subsample(train_raw, SUBSAMPLE_N)
print(f'Train cho bootstrap: {len(train_sub):,} dòng, fraud: {int(train_sub[TARGET_COL].sum())} '
      f'({train_sub[TARGET_COL].mean():.3%})')


---
## 3. Chạy Thí Nghiệm CIES
Chạy đủ 25 tổ hợp (5 model × 5 kỹ thuật). Kết quả lưu ngay sau mỗi tổ hợp nên có thể chạy lại/resume.


In [ ]:
# 3. Run CIES Experiment Runner
# Dùng N_RUNS chuẩn từ src/config.py (mặc định 20, spec mục 7.1).
# KHÔNG hardcode số run thấp hơn spec — nếu cần giảm để debug/chạy local,
# phải hỏi lại người dùng trước khi quyết định (spec mục 7.2).
CURRENT_N_RUNS = N_RUNS

cies_all_results = []

# Mỗi tổ hợp (model, technique) chạy qua run_cies_experiment_isolated() —
# tự chạy trong 1 subprocess riêng, tránh torch (ANN) và xgboost cùng tồn
# tại trong 1 process (xung đột OpenMP runtime, xem src/utils/isolation.py).
done_combos = set()
out_name = 'cies_summary_results.json'
if (RESULTS_DIR / out_name).exists():
    import json
    cies_all_results = json.load(open(RESULTS_DIR / out_name, encoding='utf-8'))
    done_combos = {(r.get('model_name'), r.get('imbalance_technique')) for r in cies_all_results if 'cies_metrics' in r}
    cies_all_results = [r for r in cies_all_results if 'cies_metrics' in r]  # bỏ bản ghi lỗi cũ để chạy lại
    print(f'Đã có {len(done_combos)} tổ hợp hợp lệ từ lần chạy trước — bỏ qua, chỉ chạy tổ hợp còn thiếu.')
    # file JSON đọc lại không còn DataFrame feature_cies — chỉ dùng để bỏ qua tổ hợp đã xong

for model_name in MODEL_NAMES:
    for technique in IMBALANCE_TECHNIQUES:
        if (model_name, technique) in done_combos:
            print(f'Bỏ qua (đã có): {model_name} x {technique}')
            continue
        print(f'\n========================================')
        print(f'Running CIES: Model={model_name}, Technique={technique}')

        try:
            res = run_cies_experiment_isolated(
                model_name=model_name,
                imbalance_technique=technique,
                df_train=train_sub,
                df_test_fixed_eval=df_test_fixed_eval,
                target_col=TARGET_COL,
                n_runs=CURRENT_N_RUNS,
                feature_level=True,
            )
            print(f'--> CIES Score: {res["cies_metrics"]["cies_score"]:.4f} (Spearman: {res["cies_metrics"]["mean_spearman"]:.4f}, runs thành công: {res["cies_metrics"]["n_runs"]})')
        except (TimeoutError, RuntimeError) as e:
            print(f'  Thất bại: {type(e).__name__}: {e}')
            res = {'model_name': model_name, 'imbalance_technique': technique, 'error': f'{type(e).__name__}: {e}'}
        cies_all_results.append(res)
        # Lưu NGAY sau mỗi tổ hợp — 1 tổ hợp lỗi/timeout không làm mất các tổ hợp đã xong
        merge_cies_result(res, RESULTS_DIR, filename=out_name)


--- 
## 4. Phân Tích Độ Ổn Định Giải Thích (CIES Analysis)
Tổng hợp kết quả bảng CIES score và vẽ Heatmap so sánh.

In [ ]:
# 4. Bảng tổng hợp CIES
summary_rows = []
for r in cies_all_results:
    if 'cies_metrics' not in r:
        continue
    summary_rows.append({
        'model': r['model_name'],
        'technique': r['imbalance_technique'],
        'cies_score': r['cies_metrics']['cies_score'],
        'mean_rank_distance': r['cies_metrics']['mean_rank_distance'],
        'std_rank_distance': r['cies_metrics']['std_rank_distance'],
        'mean_spearman': r['cies_metrics']['mean_spearman'],
    })

df_cies = pd.DataFrame(summary_rows)
print('=== BẢNG CIES STABILITY METRICS ===')
display(df_cies)

# Heatmap CIES
pivot_cies = df_cies.pivot(index='model', columns='technique', values='cies_score')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_cies, annot=True, cmap='viridis', fmt='.4f', vmin=0, vmax=1.0)
plt.title('Ma Trận CIES Score (Mô Hình × Kỹ Thuật Imbalance)', fontsize=13, fontweight='bold')
plt.ylabel('Mô Hình')
plt.xlabel('Kỹ Thuật Imbalance')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cies_heatmap.png', dpi=300)
plt.show()

--- 
## 5. Trade-off: Hiệu Năng Phân Loại (PR-AUC) vs Độ Tin Cậy Giải Thích (CIES)
Vẽ biểu đồ phân tán để tìm đường Pareto-optimal giữa PR-AUC và CIES Score.

In [ ]:
# 5. Trade-off Scatter Plot
# Ghép với kết quả PR-AUC từ notebook 03
benchmark_path = RESULTS_DIR / 'model_benchmark_results.csv'
if benchmark_path.exists():
    df_bench = pd.read_csv(benchmark_path)
    # benchmark đặt tên cột là 'imbalance_technique' — phải đổi tên, nếu không merge rơi về
    # chỉ theo 'model' (tích Descartes: ghép nhầm PR-AUC của mọi kỹ thuật).
    df_bench = df_bench.rename(columns={'imbalance_technique': 'technique'})
    df_merged = pd.merge(df_cies, df_bench, on=['model', 'technique'])
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df_merged,
        x='pr_auc',
        y='cies_score',
        hue='model',
        style='technique',
        s=150
    )
    plt.title('Trade-off: PR-AUC vs CIES Explanation Stability', fontsize=14, fontweight='bold')
    plt.xlabel('Hiệu Năng Phân Loại (PR-AUC)')
    plt.ylabel('Độ Tin Cậy Giải Thích (CIES Score)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'tradeoff_prauc_vs_cies.png', dpi=300)
    plt.show()

--- 
## 6. Feature-Level CIES (Phân Tích Cấp Đặc Trưng)
Khảo sát các đặc trưng nào có rank ổn định nhất và đặc trưng nào bị biến động mạnh nhất giữa các lần chạy.

In [ ]:
# 6. Feature-level analysis cho tổ hợp tốt nhất
best_run = max([r for r in cies_all_results if 'cies_metrics' in r], key=lambda x: x['cies_metrics']['cies_score'])
print(f'Tổ hợp có CIES cao nhất: {best_run["combination"]}')

if not isinstance(best_run.get('feature_cies'), pd.DataFrame):
    print('Không có feature_cies (kết quả nạp lại từ JSON) — chạy lại tổ hợp này trong phiên hiện tại để xem biểu đồ.')
elif 'feature_cies' in best_run and isinstance(best_run['feature_cies'], pd.DataFrame):
    df_feat = best_run['feature_cies']
    print('Top 10 features theo độ quan trọng SHAP trung bình:')
    display(df_feat.head(10))
    
    plt.figure(figsize=(10, 6))
    top_feats = df_feat.head(10)
    plt.errorbar(
        y=top_feats['feature'],
        x=top_feats['mean_abs_shap'],
        xerr=top_feats['shap_std'],
        fmt='o',
        color='darkblue',
        ecolor='red',
        elinewidth=2,
        capsize=5
    )
    plt.title(f'Độ Ổn Định Cấp Đặc Trưng - Mean |SHAP| ± Std ({best_run["combination"]})', fontsize=12, fontweight='bold')
    plt.xlabel('Mean |SHAP| Value')
    plt.ylabel('Feature')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_level_stability.png', dpi=300)
    plt.show()

--- 
## 7. Hướng Dẫn Chạy Full Quy Mô (N_RUNS = 20) Trên Kaggle / Colab
1. Upload toàn bộ thư mục `src/` lên Kaggle / Colab.
2. Giữ `N_RUNS = 20` trong `src/config.py`; `SUBSAMPLE_N` ở mục 2.
3. Sử dụng GPU/TPU accelerator để tăng tốc TreeSHAP và ANN.
4. Kết quả JSON từ `merge_cies_result()` tải về để đưa vào báo cáo luận văn.

---
## 8. Kiểm tra độ nhạy theo cỡ mẫu (chứng minh `SUBSAMPLE_N`)
Chạy CIES cho 1 tổ hợp rẻ ở nhiều cỡ mẫu. Nếu điểm CIES ngừng thay đổi rõ rệt từ một mức nào đó thì mức đó đủ lớn. Đặt `RUN_SENSITIVITY = True` để chạy.

In [ ]:
# 8. Sensitivity: CIES vs cỡ mẫu subsample
import json
RUN_SENSITIVITY = False
SENS_MODEL, SENS_TECH = 'xgboost', 'class_weighting'
SENS_SIZES = [10_000, 30_000, 50_000, 100_000]
sens_path = RESULTS_DIR / 'cies_sensitivity_subsample.json'

if RUN_SENSITIVITY:
    sens = json.load(open(sens_path, encoding='utf-8')) if sens_path.exists() else []
    done = {r['n'] for r in sens}
    for n in SENS_SIZES:
        if n in done:
            continue
        sub = stratified_subsample(train_raw, n)
        res = run_cies_experiment_isolated(
            model_name=SENS_MODEL, imbalance_technique=SENS_TECH, df_train=sub,
            df_test_fixed_eval=df_test_fixed_eval, target_col=TARGET_COL,
            n_runs=CURRENT_N_RUNS, feature_level=False)
        m = res['cies_metrics']
        sens.append({'n': n, 'n_fraud': int(sub[TARGET_COL].sum()), 'cies_score': m['cies_score'],
                     'std_rank_distance': m['std_rank_distance'], 'mean_spearman': m['mean_spearman'],
                     'n_runs': m['n_runs']})
        json.dump(sens, open(sens_path, 'w', encoding='utf-8'), indent=2)
        print(sens[-1])
if sens_path.exists():
    display(pd.DataFrame(json.load(open(sens_path, encoding='utf-8'))).sort_values('n'))
